## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

batch_size = 16
# batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
# batch_size = 512
# batch_size = 1024
# batch_size = 2048
# batch_size = 4096
# batch_size=8192
print("Misc. initializations")

2025-07-13 15:57:43.566707: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Misc. initializations


### Load and Preprocess the MNIST Dataset. 

In [2]:
# # 数据加载
# (x_train, y_train), (x_test, y_test) = octid.load_data()  # 假设已加载OCTID数据集

# # 预处理流水线
# x_train = preprocess_pipeline(x_train)  # 包含裁剪、去噪、增强等
# x_test = preprocess_pipeline(x_test)

# # 维度扩展
# x_train = np.expand_dims(x_train, -1)
# x_test = np.expand_dims(x_test, -1)

# # 标准化
# x_train = zscore_normalize(x_train)
# x_test = zscore_normalize(x_test)

In [3]:
# # 验证集构建
# x_train, x_val, y_train, y_val = stratified_split(x_train, y_train)

# print('Validation and test data ready')

# # Convert class vector to binary class matrices
# # num_classes = 10
# # y_train = utils.to_categorical(y_train, num_classes)
# # y_test = utils.to_categorical(y_test, num_classes)
# # y_val = utils.to_categorical(y_val, num_classes)

# input_shape = x_train[0].shape
# print(f'input shape: {input_shape}')

In [4]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import cv2
import os

CLASS_MAP = {
    'CSR': 0,
    'DR': 1,
    'MH': 2,
    'NORMAL': 3
}

def load_octid_dataset(dataset_path, target_size=(24, 24)):
    images, labels = [], []
    
    for class_name, class_idx in CLASS_MAP.items():
        class_dir = os.path.join(dataset_path, class_name)
        
        if not os.path.isdir(class_dir):
            raise ValueError(f"Missing class directory: {class_name}")
            
        for filename in os.listdir(class_dir):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
                
            img_path = os.path.join(class_dir, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"Warning: Failed to read {img_path}, skipping...")
                continue
                
            img = cv2.resize(img, target_size)
            img = cv2.medianBlur(img, 3)
            img = img.astype(np.float32)
            
            images.append(img)
            labels.append(class_idx)
    
    images = np.expand_dims(np.array(images), axis=-1)  # (N, 24, 24, 1)
    labels = np.array(labels, dtype=np.int32)  # 直接返回整数标签
    return images, labels

# 加载数据
dataset_path = "/workspace/examples/vision_transformer/data/OCTID"
x_all, y_all = load_octid_dataset(dataset_path)

# 分层分割函数（无需修改，但需确保stratify使用整数标签）
def stratified_split(data, labels, test_size=0.2, val_size=0.1, random_state=42):
    x_train_val, x_test, y_train_val, y_test = train_test_split(
        data, labels, test_size=test_size, stratify=labels, random_state=random_state
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_train_val, y_train_val,
        test_size=val_size/(1-test_size), stratify=y_train_val, random_state=random_state
    )
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

# 执行分割
(x_train, y_train), (x_val, y_val), (x_test, y_test) = stratified_split(x_all, y_all)

# 标签格式验证
assert y_train.ndim == 1, "训练集标签应为整数格式"
assert y_val.ndim == 1, "验证集标签应为整数格式"
assert y_test.ndim == 1, "测试集标签应为整数格式"

# 标准化处理（原有代码不变）
train_mean = np.mean(x_train, axis=(0,1,2))
train_std = np.std(x_train, axis=(0,1,2))

def zscore_normalize(images, mean, std):
    return (images - mean) / (std + 1e-7)

x_train = zscore_normalize(x_train, train_mean, train_std)
x_val = zscore_normalize(x_val, train_mean, train_std)
x_test = zscore_normalize(x_test, train_mean, train_std)

# 输出验证
print(f"Input shape: {x_train[0].shape}")  # (24, 24, 1)
print(f"y_train shape: {y_train.shape}")  # 应为 (样本数,)
print(f"标签示例: {y_train[:5]}")  # 如 [0, 1, 3, 2, ...]

Input shape: (24, 24, 1)
y_train shape: (361,)
标签示例: [1 3 3 3 2]


### Save Dataset

In [5]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test7')
save_data_set(x_train, y_train, data_type='train')
save_data_set(x_val, y_val, data_type='val')

Saving x_test7 of shape (104, 24, 24, 1) in data/net_mnist/x_test7.h5
Saving x_train of shape (361, 24, 24, 1) in data/net_mnist/x_train.h5
Saving x_val of shape (52, 24, 24, 1) in data/net_mnist/x_val.h5


In [6]:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# datagen = ImageDataGenerator(
#     rotation_range=10,
#     width_shift_range=0.1,
#     height_shift_range=0.1,
#     zoom_range=0.1
# )


### Build a Plain Neural Network for the MNIST Model

In [7]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D,Softmax
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2

model = Sequential()

# 输入层修改为 (24,24,1)
model.add(Conv2D(20, (3, 3), input_shape=(24,24,1), kernel_regularizer=l2(0.0005)))  # (22,22,20)

# AveP-BN操作
model.add(AvgPool2D(pool_size=2))  # (11,11,20)
model.add(BatchNormalization())

# 第二层卷积层（无激活函数）
model.add(Conv2D(50, (3, 3), kernel_regularizer=l2(0.0005)))  # (9,9,50)

# 调整后的BN + ReLU顺序
model.add(ReLU())  # 激活函数前置
model.add(BatchNormalization())

# 1x1卷积调整通道数到4
model.add(Conv2D(4, (1, 1), kernel_regularizer=l2(0.0005)))  # (9,9,4)

# 替换方案：平均池化 + 展平
model.add(AvgPool2D(pool_size=(9,9), strides=(9,9)))  # 池化窗口覆盖整个空间维度，输出 (1,1,4)
model.add(Flatten())  # 展平为4维向量

# 输出模型架构
model.summary()

2025-07-13 15:57:51.136577: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.176301: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.179198: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.182429: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the approp

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 20, 20, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 10, 10, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 6, 6, 50)          25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 3, 3, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 4)           1804      
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 4)          1

urning NUMA node zero
2025-07-13 15:57:51.185816: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.188322: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.936093: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.937906: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:57:51.939477: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful

### 第一阶段

In [8]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=20,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
# optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
def lr_scheduler(epoch):
    if epoch < 100:
        return 0.001
    elif epoch < 200:
        return 0.0005
    else:
        return 0.0001
callbacks = [
    tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000


/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make su

23/23 - 3s - loss: 1.3217 - accuracy: 0.4238 - val_loss: 1.1405 - val_accuracy: 0.4808 - lr: 0.0010 - 3s/epoch - 133ms/step
Epoch 2/4000
23/23 - 0s - loss: 1.1807 - accuracy: 0.5319 - val_loss: 1.0545 - val_accuracy: 0.5577 - lr: 0.0010 - 88ms/epoch - 4ms/step
Epoch 3/4000
23/23 - 0s - loss: 1.1518 - accuracy: 0.5845 - val_loss: 1.0405 - val_accuracy: 0.6154 - lr: 0.0010 - 87ms/epoch - 4ms/step
Epoch 4/4000
23/23 - 0s - loss: 1.0855 - accuracy: 0.6150 - val_loss: 1.1149 - val_accuracy: 0.6154 - lr: 0.0010 - 84ms/epoch - 4ms/step
Epoch 5/4000
23/23 - 0s - loss: 1.0826 - accuracy: 0.6343 - val_loss: 1.0840 - val_accuracy: 0.6346 - lr: 0.0010 - 85ms/epoch - 4ms/step
Epoch 6/4000
23/23 - 0s - loss: 1.0438 - accuracy: 0.6094 - val_loss: 1.1118 - val_accuracy: 0.5962 - lr: 0.0010 - 83ms/epoch - 4ms/step
Epoch 7/4000
23/23 - 0s - loss: 1.0254 - accuracy: 0.6150 - val_loss: 1.0510 - val_accuracy: 0.6538 - lr: 0.0010 - 84ms/epoch - 4ms/step
Epoch 8/4000
23/23 - 0s - loss: 1.0173 - accuracy: 0.6

In [9]:
# model.get_weights()

### Rebuild Model

In [10]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
new_model = Sequential()
for layer in model.layers:
    
    if isinstance(layer, ReLU):
        new_model.add(SquareActivation())
    else:
        new_layer = layer.__class__.from_config(layer.get_config())
        new_model.add(new_layer)
        if layer.weights:
            new_layer.set_weights(layer.get_weights())  # 自动复制权重和偏置

new_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 20, 20, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 10, 10, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 6, 6, 50)          25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 3, 3, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 4)           1804      
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 4)         

In [11]:
# new_model.get_weights()

第二阶段

In [12]:
#二阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=20,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
# optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
def lr_scheduler(epoch):
    if epoch < 100:
        return 0.001
    elif epoch < 200:
        return 0.0005
    else:
        return 0.0001
callbacks = [
    tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

new_model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

new_model.fit(
    x_train, y_train, batch_size=batch_size,
   
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True,
              callbacks=callbacks
              )

score = new_model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000
23/23 - 1s - loss: 0.5745 - accuracy: 0.8283 - val_loss: 0.7701 - val_accuracy: 0.6923 - lr: 0.0010 - 696ms/epoch - 30ms/step
Epoch 2/4000
23/23 - 0s - loss: 0.5585 - accuracy: 0.8283 - val_loss: 0.7392 - val_accuracy: 0.6538 - lr: 0.0010 - 92ms/epoch - 4ms/step
Epoch 3/4000
23/23 - 0s - loss: 0.5623 - accuracy: 0.8615 - val_loss: 0.8064 - val_accuracy: 0.6923 - lr: 0.0010 - 92ms/epoch - 4ms/step
Epoch 4/4000
23/23 - 0s - loss: 0.5499 - accuracy: 0.8449 - val_loss: 0.7527 - val_accuracy: 0.6923 - lr: 0.0010 - 90ms/epoch - 4ms/step
Epoch 5/4000
23/23 - 0s - loss: 0.5463 - accuracy: 0.8532 - val_loss: 0.7432 - val_accuracy: 0.7115 - lr: 0.0010 - 90ms/epoch - 4ms/step
Epoch 6/4000
23/23 - 0s - loss: 0.5446 - accuracy: 0.8587 - val_loss: 0.7836 - val_accuracy: 0.6731 - lr: 0.0010 - 87ms/epoch - 4ms/step
Epoch 7/4000
23/23 - 0s - loss: 0.5106 - accuracy: 0.8504 - val_loss: 0.7245 - val_accuracy: 0.7308 - lr: 0.0010 - 91ms/epoch - 4ms/step
Epoch 8/4000
23/23 - 0s - loss: 0.5200 

### Serialize Model and Weights

In [13]:
model_json = new_model.to_json()
with open(os.path.join(PATH, 'model411.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
new_model.save_weights(os.path.join(PATH, 'model411.h5'))
print("Saved model to disk")

Saved model to disk


******************************************************************************************
We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [ ]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

In [ ]:
import pyhelayers
import utils
import h5py
import os
import numpy as np
##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
# from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)



utils.verify_memory()

print('Misc. initializations')

batch_size=500

The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [ ]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [ ]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [ ]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [ ]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
